# Классы и ООП

## Цели лекции

После этой лекции вы сможете:
- определять классы, атрибуты и методы; создавать экземпляры;
- применять наследование и специализацию (переопределение) методов, абстрактные классы;
- понимать пространства имён классов и внутреннее устройство объектов.

### Зачем нужны классы

До сих пор мы **пользовались** готовыми объектами – списками, словарями, файлами –
и вызывали их методы, как `data.append(4)` ниже. Такой стиль называют
объектно-основанным. Классы позволяют создавать **собственные** типы объектов
и связывать их в иерархию – это уже объектно-ориентированный стиль.

- **Новые типы под предметную область**: робот, документ, заказ – со своими данными
  и своим поведением в одном объекте.
- **Наследование**: новый класс берёт поведение старого и переопределяет только то,
  что нужно изменить. Исходный код при этом не трогают.
- **Композиция**: объект собирается из других объектов как из частей.
- **Полиморфизм**: одинаковый вызов работает по-разному в зависимости от типа объекта.
- **Инкапсуляция**: детали реализации скрыты за методами класса. В Python – по
  соглашению (имя с `_`), а не запретом доступа.

Пример, к которому будем возвращаться: робот, готовящий пиццу. Он **наследует**
поведение обычного робота (двигаться, выполнять команды) и **состоит** из частей –
руки и мотора.

На этом принципе построены и готовые библиотеки: работа с базами данных, веб-фреймворки,
тестирование – везде новый класс наследует от базового и реализует несколько своих методов.


In [14]:
# Объектно-основанный стиль без классов
data = [1, 2, 3]
data.append(4)  # используем методы уже существующих объектов
total = sum(data)
print(total)

10


In [15]:
class Counter:
    def __init__(self, start=0):
        self.value = start  # атрибут экземпляра

    def inc(self, step=1):
        self.value += step  # изменение состояния

c = Counter()
c.inc()
print(c.value)

1


In [16]:
# Пример наследования и композиции

class Robot:
    def move(self):
        print("Робот движется")

class Arm:
    def roll_dough(self):
        print("Рука раскатывает тесто")

class Motor:
    def drive(self):
        print("Мотор вращается")

class PizzaRobot(Robot):  # наследуем свойства обычного робота
    def __init__(self):
        self.arm = Arm()
        self.motor = Motor()

    def make_pizza(self):
        print("Начало приготовления пиццы...")
        self.arm.roll_dough()
        self.motor.drive()
        print("Пицца готова!")

bot = PizzaRobot()
bot.move()        # поведение, унаследованное от Robot
bot.make_pizza()  # использование композиции


Робот движется
Начало приготовления пиццы...
Рука раскатывает тесто
Мотор вращается
Пицца готова!


### Как Python ищет атрибут

Вся механика наследования сводится к одному выражению: `объект.атрибут`.

- Для обычных атрибутов Python сначала проверяет экземпляр, затем класс и его
  предков в определённом порядке – он называется MRO, о нём в разделе про
  наследование. Исключение – `property`: свойство класса проверяется раньше
  экземпляра. К свойствам вернёмся после `__init__`.
- Первое найденное имя выигрывает: если подкласс определил атрибут, он **замещает**
  одноимённый атрибут родителя.

Словарь, который понадобится дальше:

- **класс** – описание, по которому создаются экземпляры; хранит общие атрибуты и методы;
- **экземпляр** – конкретный объект со своими данными;
- **подкласс** (ниже по дереву) и **суперкласс** (выше).


In [17]:
# Пример дерева наследования и поиска атрибутов

class C3:
    w = "из C3"
    z = "из C3"

class C2:
    z = "из C2"
    x = "из C2"

class C1(C2, C3):
    x = "из C1"
    y = "из C1"

# создаём экземпляры, связанные с классом
I1 = C1()
I2 = C1()
I2.name = "I2"

# Проверяем, где Python находит атрибуты: экземпляр → C1 → C2 → C3
print("I2.w =", I2.w)        # нет ни в экземпляре, ни в C1, ни в C2 – найдено в C3
print("I1.x =", I1.x)        # есть в C1 и в C2: C1 ближе, его x замещает C2.x
print("I2.y =", I2.y)        # найдено в C1
print("I1.z =", I1.z)        # есть в C2 и в C3: C2 стоит в скобках левее
print("I2.name =", I2.name)  # найдено прямо в экземпляре

I2.w = из C3
I1.x = из C1
I2.y = из C1
I1.z = из C2
I2.name = I2


In [30]:
# Переиспользование кода через наследование и полиморфизм

class Employee:  # базовый класс
    def compute_salary(self):
        print("Вычисление зарплаты (общая формула)")

    def give_raise(self):
        print("Повышение зарплаты")

class Engineer(Employee):  # подкласс с переопределённым методом
    def compute_salary(self):
        print("Вычисление зарплаты для инженера (особая формула)")

# создаём объекты разных типов
ivan = Employee()
petr = Employee()
stas = Engineer()

# все объекты вызывают одинаковый метод, но поведение различается
company = [ivan, petr, stas]

for emp in company:
    emp.compute_salary()  # автоматически выбирается нужная версия метода

Вычисление зарплаты (общая формула)
Вычисление зарплаты (общая формула)
Вычисление зарплаты для инженера (особая формула)


In [3]:
# Программирование через кастомизацию (наследование из готового кода)

class FrameworkReader:
    def read(self):
        raise NotImplementedError("Метод read() должен быть переопределён")

class FileReader(FrameworkReader):
    def read(self):
        print("Чтение данных из файла")

class NetworkReader(FrameworkReader):
    def read(self):
        print("Чтение данных из сети")

# "Фреймворк" вызывает методы, реализованные пользователем
def process_data(reader):
    reader.read()

process_data(FileReader())    # Чтение данных из файла
process_data(NetworkReader()) # Чтение данных из сети


Чтение данных из файла
Чтение данных из сети


### Классы и экземпляры

- Класс и экземпляры являются **пространствами имён**: в них можно хранить данные и функции (атрибуты).  
- Классы похожи на **модули**, но могут порождать множество экземпляров, тогда как модуль существует в единственном экземпляре.

Главное различие:
- Класс описывает **что умеет объект делать** (например, методы `compute_salary`, `give_raise`).  
- Экземпляр хранит **конкретные данные**, с которыми эти методы работают (например, имя сотрудника, часы работы, ставка).


### Вызов методов

- Методы – это функции, определённые внутри класса.  
- Они всегда вызываются **через экземпляр** или **через сам класс**.  
- При вызове метода Python **автоматически передаёт** в него объект, на котором метод был вызван.  
  Этот объект поступает в первый параметр метода, который по соглашению называется `self`.

Например, при вызове `ivan.give_raise(10)` Python выполняет два шага:
1. Ищет метод `give_raise` в экземпляре `ivan` и его классах (наследование).  
2. Передаёт сам объект `ivan` как первый аргумент функции.

Вызовы `ivan.give_raise(10)` и `Employee.give_raise(ivan, 10)` эквивалентны.


In [20]:
class Employee:
    def __init__(self, name, pay):
        self.name = name
        self.pay = pay

    def give_raise(self, percent):
        self.pay *= (1 + percent / 100)
        print(f"{self.name}: новая зарплата {self.pay:.2f}")

# Создаём объект
ivan = Employee("Иван", 150000)

# Вариант 1 – обычный вызов через экземпляр
ivan.give_raise(10)

# Вариант 2 – вызов через класс (эквивалентный).
# Зарплата уже повышена первым вызовом: 5 % считаются от 165000
Employee.give_raise(ivan, 5)

Иван: новая зарплата 165000.00
Иван: новая зарплата 173250.00


### Построение деревьев классов и создание экземпляров

- Каждая инструкция `class` создаёт **новый объект класса**.  
- Каждый вызов класса (через круглые скобки) создаёт **новый экземпляр (объект)**.  
- Экземпляры **автоматически связаны** с тем классом, из которого созданы.  
- Классы **автоматически связываются со своими родителями** в порядке, указанном в скобках при объявлении (`class C1(C2, C3):`).

Таким образом, создавая классы и экземпляры, мы формируем **дерево наследования**:  
экземпляры – на нижнем уровне, классы – выше, их родительские классы – ещё выше.

In [22]:
# Пример создания дерева классов и экземпляров

class C2: ...
class C3: ...
class C1(C2, C3): ...  # класс C1 наследует C2 и C3 (в этом порядке)

# Создание экземпляров (объектов)
I1 = C1()
I2 = C1()

print(isinstance(I1, C1))   # I1 создан классом C1
print(issubclass(C1, C2))   # C1 – подкласс C2
print(issubclass(C1, C3))   # и подкласс C3: родителей может быть несколько

True
True
True


### Присвоение и область видимости атрибутов

- Атрибуты можно «вешать» как на класс, так и на экземпляр.  
- **Атрибуты класса** общие для всех его экземпляров и подклассов.  
- **Атрибуты экземпляра** принадлежат только конкретному объекту.  
- Где выполняется присвоение, там и появляется атрибут:  
  - В теле класса – создаётся **атрибут класса**.  
  - Внутри метода через `self` – создаётся **атрибут экземпляра**.
- В теле класса атрибут экземпляра создать нельзя: тело выполняется один раз,
  при создании класса, и экземпляров ещё нет. Атрибуты экземпляра появляются только
  через `self` в методах (обычно в `__init__`) или снаружи: `obj.x = ...`.


In [25]:
# Присвоение атрибутов через self
# C2 и C3 – те же, что выше

class C1(C2, C3):
    class_name = "class"  # создаёт атрибут класса

    def setname(self, who):
        self.name = who  # создаёт атрибут экземпляра

# Создание экземпляров
I1 = C1()
I2 = C1()

I1.setname("Иван")
I2.setname("Петр")

print(I1.name)
print(I2.name)

Иван
Петр


In [28]:
# В экземпляре атрибута class_name нет: Python находит его в классе C1
I2.class_name

'class'

### Конструктор `__init__`

- `__init__` уже встречался в примерах выше; теперь разберём, как он работает.
- В примере с `setname` атрибут `name` появлялся только после вызова этого метода.
  Если обратиться к нему раньше – будет ошибка.
- Чтобы атрибуты гарантированно появлялись при создании объекта, используется
  метод `__init__`: Python вызывает его автоматически сразу после создания экземпляра.
- Первый параметр – `self`, остальные приходят из скобок при вызове класса.

`__init__` – один из **специальных методов**: их имена начинаются и заканчиваются
двойным подчёркиванием, и Python вызывает их сам – при создании объекта, при `+`,
`==`, `len()` и других встроенных операциях. Методы для операторов (перегрузка
операторов) разбираются в лекции 7б.


In [29]:
# Пример конструктора __init__
# C2 и C3 – те же, что выше

class C1(C2, C3):
    def __init__(self, who):
        self.name = who  # создаётся при создании объекта

# При создании экземпляра вызывается __init__
I1 = C1("Иван")
I2 = C1("Петр")

print(I1.name)
print(I2.name)

Иван
Петр


### `__init__` не должен возвращать созданный объект

`return self` в `__init__` вызывает `TypeError`: допустим только результат `None`.
Обычно метод просто заканчивается без `return`. Новый экземпляр класса
создаёт специальный метод `__new__`, а `__init__` лишь заполняет уже созданный
экземпляр полями. Вызов `ReadyItem("Тетрадь")` проходит оба шага и возвращает
готовый экземпляр. О `__new__` – в лекции 7б.

In [1]:
class BadInit:
    def __init__(self):
        return self

try:
    BadInit()
except TypeError:
    print("__init__ должен возвращать None")

class ReadyItem:
    def __init__(self, name):
        self.name = name

item = ReadyItem("Тетрадь")
print(item.name)

__init__ должен возвращать None
Тетрадь


### Объект должен сохранять допустимое состояние

Пусть вместимость и число бронируемых мест – целые числа. У объекта есть правило:
остаток мест никогда не бывает отрицательным. Правило, которое должно выполняться
всегда – сразу после создания объекта и после любого его метода, – называют
**инвариантом**. Чтобы он не нарушался, проверяем начальное значение и каждый
запрос **до изменения поля**.

In [1]:
class SeatPool:
    def __init__(self, capacity):
        if capacity < 0:
            raise ValueError("Вместимость не может быть отрицательной")
        self.remaining = capacity

    def reserve(self, count):
        if count <= 0 or count > self.remaining:
            raise ValueError("Недопустимое число мест")
        self.remaining -= count

pool = SeatPool(3)
pool.reserve(2)
print("Осталось мест:", pool.remaining)

Осталось мест: 1


### Ошибка запроса не должна портить объект

Проверим отказ, затем продолжим работу с тем же объектом.
Правило соблюдают наши методы; прямое присваивание `pool.remaining`
в обход методов пока ничем не ограничено.

In [2]:
try:
    pool.reserve(2)
except ValueError as error:
    print(error)
print("После отказа:", pool.remaining)

pool.reserve(1)
print("После успешного запроса:", pool.remaining)

try:
    SeatPool(-1)
except ValueError as error:
    print(error)

Недопустимое число мест
После отказа: 1
После успешного запроса: 0
Вместимость не может быть отрицательной


### Вычисляемый атрибут через `@property`

Площадь зависит от текущих сторон. Вместо отдельного поля с сохранённой
площадью определим вычисление: `@property` позволяет читать результат
метода как атрибут, без скобок.

Зачем: внешний код читает `panel.area` одинаково – и когда это обычное поле,
и когда свойство. Поэтому можно начать с простого поля, а позже заменить его
вычислением, не трогая код, который читает площадь:

```python
# было: поле, посчитанное один раз в __init__ (устареет при смене сторон)
self.area = width * height

# стало: свойство – чтение panel.area по-прежнему без скобок
@property
def area(self):
    return self.width * self.height
```

С обычным методом `area()` пришлось бы дописать скобки во всех местах чтения.

In [3]:
class Panel:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

panel = Panel(3, 4)
print(panel.area)
panel.width = 5
print(panel.area)

12
20


### Свойство без сеттера доступно только для чтения

У нашего `area` есть только способ вычисления, функции записи (сеттера) нет.

Обычно `obj.x = ...` записывает значение в атрибуты экземпляра. Но если в классе
есть свойство с таким именем, Python сначала находит его в классе и передаёт
присваивание свойству – оно и решает, что делать. Сеттера у `area` нет,
поэтому присваивание падает с `AttributeError`, а поле `area` в экземпляре
не появляется. Чтение `panel.area` тоже всегда идёт через свойство.

Менять нужно стороны; остальной объект от этого не становится неизменяемым.
Для дорогого вычисления или действия с побочным эффектом лучше явный метод.

In [4]:
try:
    panel.area = 100
except AttributeError as error:
    print("AttributeError:", error)

panel.height = 2
print("Новая площадь:", panel.area)
print("Сохранённые поля:", vars(panel))   # vars – словарь атрибутов экземпляра, area там нет

AttributeError: property 'area' of 'Panel' object has no setter
Новая площадь: 10
Сохранённые поля: {'width': 5, 'height': 2}


### Сеттер проверяет присваивание атрибуту

Свойство может управлять и записью. Пусть громкость задана целым числом от 0 до 100;
значение храним в `_volume` (одинарное `_` – внутреннее поле по соглашению).

`@volume.setter` берёт уже созданное свойство `volume` и добавляет ему функцию
записи, поэтому оба метода называются одинаково. Присваивание `self.volume = volume`
в `__init__` тоже идёт через сеттер: проверка срабатывает уже при создании объекта.

In [1]:
class Speaker:
    def __init__(self, volume):
        self.volume = volume      # через сеттер: проверка и при создании

    @property
    def volume(self):
        return self._volume

    @volume.setter
    def volume(self, value):
        if not 0 <= value <= 100:
            raise ValueError("Громкость должна быть от 0 до 100")
        self._volume = value

speaker = Speaker(20)
speaker.volume = 60
print(speaker.volume)

60


### Почему сеттер записывает в другое имя

Любое присваивание `obj.volume = ...` идёт через сеттер – и снаружи, и внутри
класса. Если сам сеттер пишет `self.volume = value`, он вызывает сам себя,
тот – снова себя, и так без конца. Это рекурсия без базового случая (лекция 5),
она обрывается `RecursionError`.

Поэтому значение хранят под другим именем `_volume`: свойства с таким именем нет,
и запись идёт прямо в экземпляр. Проверка перед записью сохраняет прежнее
значение при отказе.

In [2]:
# Ошибка: сеттер пишет в своё же имя и вызывает сам себя
class BrokenSpeaker:
    @property
    def volume(self):
        return self._volume

    @volume.setter
    def volume(self, value):
        self.volume = value        # снова вызывает этот же сеттер

try:
    BrokenSpeaker().volume = 50
except RecursionError:
    print("RecursionError: сеттер вызывал сам себя")

# Speaker пишет в _volume, поэтому работает
try:
    speaker.volume = 150
except ValueError as error:
    print(error)
print("После отказа:", speaker.volume)

try:
    Speaker(-1)
except ValueError:
    print("Недопустимая начальная громкость отклонена")

speaker.volume = 0
print("Нижняя граница:", speaker.volume)

RecursionError: сеттер вызывал сам себя
Громкость должна быть от 0 до 100
После отказа: 60
Недопустимая начальная громкость отклонена
Нижняя граница: 0


### Классы как атрибуты модулей

- Имя класса – это **обычная переменная**, которая ссылается на объект класса,
  созданный инструкцией `class`.  
- Классы всегда определяются **внутри модулей**,  
  и потому подчиняются тем же правилам, что и функции или переменные.  
- В одном модуле можно разместить **несколько классов, функций и переменных** –  
  они все становятся атрибутами этого модуля.  


In [31]:
# Пример структуры модуля names.py

var1 = 6
var2 = 3.12

def func1():
    pass

def func2():
    pass

class Cls1:
    pass

class Cls2:
    pass

# После импорта модуля все имена доступны как атрибуты:
# names.var1, names.func1, names.Cls1 и т.д.

### Самый простой класс в Python

- В Python можно создать **пустой класс**, не содержащий никаких методов и атрибутов.  
- Такой класс служит **пустым пространством имён**, в которое можно добавлять атрибуты уже после создания.  
- Тело класса не может быть пустым, поэтому в нём пишут инструкцию `pass` – она ничего не делает.  
- Класс можно использовать как простой контейнер данных.

In [35]:
# Самый простой класс

class Record:
    pass  # пустой класс, пока без атрибутов и методов

# Добавляем атрибуты классу после его создания
Record.name = "Иван"
Record.age = 40

print(Record.name, Record.age)

Иван 40


In [36]:
# Создаём экземпляры класса
x = Record()
y = Record()

print(x.name, y.name)    # своих атрибутов у экземпляров нет – берутся из класса

# Изменяем атрибут только у одного экземпляра
x.name = "Фёдор"
print(Record.name, x.name, y.name)   # у x теперь своё name, класс и y не изменились

Иван Иван
Иван Фёдор Иван


### Классы изнутри: внутреннее устройство

- В Python классы и экземпляры реализованы как **связанная система пространств имён**, где каждое пространство хранится во внутреннем словаре (`__dict__`).  
- Атрибуты классов и объектов фактически хранятся в этих словарях, а механизм наследования – это просто **поиск в цепочке связанных словарей**.  
- Каждый экземпляр имеет ссылку на свой класс через `__class__`,  
  а каждый класс – на своих родителей через `__bases__`.  
- Это делает модель Python **очень динамичной**:  
  атрибуты, методы и даже связи между классами можно изменять в любое время, уже после создания объектов.  


In [37]:
# Исследуем внутренние словари (__dict__) класса и его экземпляров.
# В словаре класса Python хранит и служебные имена с __, покажем только наши
print({key: value for key, value in Record.__dict__.items() if not key.startswith("__")})
print(x.__dict__)   # у x своё имя name
print(y.__dict__)   # у y своих атрибутов нет

{'name': 'Иван', 'age': 40}
{'name': 'Фёдор'}
{}


In [43]:
# Ссылки между объектами
print(x.__class__)        # экземпляр знает свой класс
print(Record.__bases__)   # родители класса: object – общий предок всех классов

<class '__main__.Record'>
(<class 'object'>,)


In [44]:
# Добавляем функцию как метод после создания класса
def uppername(obj):
    return obj.name.upper()

Record.method = uppername  # функция становится методом класса

In [45]:
print(x.method())         # self – это x, у x своё имя «Фёдор»
print(y.method())         # self – это y, своего name нет, берётся Record.name
print(Record.method(x))   # то же, что x.method(): экземпляр передан явно

ФЁДОР
ИВАН
ФЁДОР


### Классы как замена словарям и записям

Данные об объекте мы до сих пор хранили в кортежах и словарях. Экземпляр класса
хранит их в атрибутах: `person.name` вместо `person[0]` или `person["name"]`.
Имя поля видно в коде, и редактор подсказывает атрибуты класса, а строковый ключ
словаря он обычно не проверяет.

Для классов, которые только хранят данные, есть `dataclass` – он в лекции 7б.

### Методы в классах

Кроме обычных методов (экран «Вызов методов»), в классе бывают **методы класса**
(`@classmethod`) и **статические методы** (`@staticmethod`). Различаются они тем,
что получают первым аргументом:

| | обычный метод | `@classmethod` | `@staticmethod` |
| --- | --- | --- | --- |
| первый аргумент | экземпляр `self` | класс `cls` | нет |
| что знает | данные объекта и его класса | класс, через который вызван, в том числе подкласс | только свои аргументы |
| типичная задача | работа с состоянием объекта | второй способ создания объекта (`from_string`) | вспомогательная операция по смыслу класса |


### `@classmethod`: создание объекта из строки

Обычный вызов принимает готовые значения. Второй способ создания может
разобрать строку и передать результат тому же `__init__`. Метод, который
создаёт и возвращает новый объект, называют **фабричным**.
Первый аргумент метода класса – `cls`, класс, для которого вызван метод.

In [5]:
class StudyGroup:
    def __init__(self, name, size):
        self.name = name
        self.size = size

    @classmethod
    def from_string(cls, text):
        name, size = text.split(":")
        return cls(name, int(size))

group = StudyGroup.from_string("ПИ-21:25")
print(group.name, group.size)

ПИ-21 25


### Почему фабричный метод возвращает `cls(...)`

`cls` – это класс, через который вызван метод. Поэтому `cls(...)` создаёт объект
именно этого класса, и подкласс наследует способ создания без копирования
разбора строки.

Если бы метод возвращал `StudyGroup(name, int(size))`, вызов через подкласс
создал бы объект базового класса – разница видна в коде ниже. Приём работает,
пока подкласс принимает те же аргументы конструктора, что и базовый класс.

In [6]:
class EveningGroup(StudyGroup):
    pass

evening = EveningGroup.from_string("ПИ-22:18")
print(type(evening).__name__)          # EveningGroup: cls – это EveningGroup
print(evening.name, evening.size)
print(isinstance(evening, StudyGroup))


# Для сравнения: класс записан в методе жёстко
class FixedGroup(StudyGroup):
    @classmethod
    def from_string(cls, text):
        name, size = text.split(":")
        return StudyGroup(name, int(size))   # не cls, а конкретный класс

class EveningFixed(FixedGroup):
    pass

print(type(EveningFixed.from_string("ПИ-23:15")).__name__)   # StudyGroup – не тот класс

EveningGroup
ПИ-22 18
True
StudyGroup


### `@staticmethod`: операция без `self` и `cls`

Приведение шифра студента к единому виду не требует состояния объекта или класса.
Статический метод связывает такую операцию с типом по смыслу;
если этой связи нет, достаточно обычной функции модуля.

In [7]:
class StudentCode:
    def __init__(self, raw):
        self.value = self.normalize(raw)

    @staticmethod
    def normalize(raw):
        return raw.strip().upper()

print(StudentCode.normalize("  ab-12 "))
student_code = StudentCode("  cd-34 ")
print(student_code.value)

AB-12
CD-34


### Наследование в классах

Как Python ищет атрибут в экземпляре и классах, разбирали в начале лекции
(экран «Как Python ищет атрибут»).
Порядок классов для поиска атрибутов называют **MRO** (Method Resolution Order),
он хранится в `Class.__mro__`.

- При одиночном наследовании это цепочка от подкласса к его предкам.
- При множественном наследовании Python согласует порядок родителей:
  подкласс располагается раньше своих предков, каждый класс встречается один раз.
- Это не простой обход дерева в глубину слева направо. Ромбовидную
  иерархию разберём после примеров специализации методов.


### Специализация унаследованных методов

- Подклассы могут:
  - **Переопределять методы** родительского класса.  
  - **Расширять поведение**, добавляя новые действия и вызывая следующую реализацию.  
- Вызов другой реализации метода:
  - `Super.method(self)` – обратиться к конкретно указанному классу.  
  - `super().method()` – продолжить поиск по MRO; это позволяет методам
    нескольких классов согласованно расширять поведение.  
- На практике пишут `super()`; явное имя класса нужно, только когда требуется
  реализация именно этого класса.  
- Если переопределённый `__init__` должен сохранить базовую инициализацию,
  её нужно вызвать явно, обычно через `super().__init__(...)`.  


In [47]:
# Пример переопределения и расширения методов суперкласса

class Super:
    def method(self):
        print("Super.method")

class Sub(Super):
    def method(self):
        print("Sub.method: начало")
        Super.method(self)   # явное имя класса; обычно пишут super().method()
        print("Sub.method: конец")

x = Super()
x.method()

y = Sub()
y.method()

Super.method
Sub.method: начало
Super.method
Sub.method: конец


In [48]:
# Пример расширения конструктора
class Super:
    def __init__(self, x):
        print("Super.__init__: базовая часть")
        self.x = x

class Sub(Super):
    def __init__(self, x, y):
        super().__init__(x)      # сначала инициализируем родительскую часть
        print("Sub.__init__: своя часть")
        self.y = y

i = Sub(1, 2)
print(i.x, i.y)

Super.__init__: базовая часть
Sub.__init__: своя часть
1 2


### Родительский конструктор может вызвать метод подкласса

Объект создаёт вызов `DetailedReport(...)` – это экземпляр класса `DetailedReport`.
`super().__init__(title)` передаёт базовому `__init__` тот же самый объект, нового
не создаётся. Поэтому внутри `BaseReport.__init__` переменная `self` – по-прежнему
объект `DetailedReport`, и `self.make_heading()` находит переопределённый метод.

А поле `author` ещё не создано: строка `self.author = author` выполнится только
после возврата из базового `__init__`.

In [2]:
class BaseReport:
    def __init__(self, title):
        print("в BaseReport.__init__ self – объект класса", type(self).__name__)
        self.title = title
        self.heading = self.make_heading()

    def make_heading(self):
        return self.title

class DetailedReport(BaseReport):
    def __init__(self, title, author):
        super().__init__(title)
        self.author = author

    def make_heading(self):
        return f"{self.title} / {self.author}"

try:
    DetailedReport("Практика", "Анна")
except AttributeError:
    print("Метод подкласса вызван, но author ещё не создан")

в BaseReport.__init__ self – объект класса DetailedReport
Метод подкласса вызван, но author ещё не создан


### Вызываем метод после завершения инициализации

В исправленном варианте конструкторы только создают свои поля.
Формирование заголовка вызывается после успешного создания объекта:
так метод получает уже подготовленное состояние.

In [3]:
class BaseReportFixed:
    def __init__(self, title):
        self.title = title

    def make_heading(self):
        return self.title

class DetailedReportFixed(BaseReportFixed):
    def __init__(self, title, author):
        super().__init__(title)
        self.author = author

    def make_heading(self):
        return f"{self.title} / {self.author}"

report = DetailedReportFixed("Практика", "Анна")
print(report.make_heading())

Практика / Анна


### MRO: один порядок для ромба наследования

Общий предок не появляется в порядке поиска дважды. Посмотрим `__mro__`
для ромба: `Combined` наследует `Left` и `Right`, оба наследуют `Root`.
Это порядок поиска методов, а не автоматический вызов всех найденных методов.

In [8]:
class Root:
    def route(self):
        return ["Root"]

class Left(Root):
    def route(self):
        return ["Left"] + super().route()

class Right(Root):
    def route(self):
        return ["Right"] + super().route()

class Combined(Left, Right):
    pass

print([cls.__name__ for cls in Combined.__mro__])

['Combined', 'Left', 'Right', 'Root', 'object']


### `super()` продолжает поиск после текущего класса

В методе `Left.route` для объекта `Combined` следующий класс по MRO – `Right`,
хотя непосредственный родитель `Left` – `Root`.
Для обычного объекта `Left` следующий класс уже `Root`.

Правило: `super()` – не «родитель класса, где написан метод», а «следующий класс
в MRO того объекта, для которого метод вызван».

In [9]:
# Combined().route(): Left.route → super() по MRO Combined → Right.route
# → super() → Root.route; списки складываются на обратном пути
print("Combined:", Combined().route())

# У объекта Left свой MRO – Left, Root: после Left сразу Root
print("Left:", Left().route())

Combined: ['Left', 'Right', 'Root']
Left: ['Left', 'Root']


### Абстрактные суперклассы (Abstract Superclasses)

- **Абстрактный суперкласс** – это класс, который задаёт общую структуру поведения,  
  но **ожидает реализации некоторых методов в подклассах**.  
- Если подкласс не определяет требуемый метод, возникает ошибка при обращении.  
- Такой подход используется во фреймворках: родитель задаёт каркас, а потомки реализуют детали.  
- Такой метод в базовом классе обычно состоит из одной строки:
  `raise NotImplementedError('сообщение')`.  
- Более строгий вариант – модуль `abc`:  
  - класс наследуют от `ABC`;  
  - обязательные методы помечают декоратором `@abstractmethod`;  
  - экземпляр нельзя даже создать, пока не реализованы все абстрактные методы.


In [50]:
# Пример простого абстрактного суперкласса
class Super:
    def delegate(self):
        self.action()

    def action(self):
        raise NotImplementedError("Метод action() должен быть определён в подклассе!")

class Sub(Super):
    def action(self):
        print("action определён в подклассе")

x = Sub()
x.delegate()

class Incomplete(Super):
    pass                       # action не определён

y = Incomplete()               # объект создаётся без ошибки
try:
    y.delegate()
except NotImplementedError as error:
    print("при вызове:", error)

action определён в подклассе
при вызове: Метод action() должен быть определён в подклассе!


In [51]:
# Пример с использованием abc
from abc import ABC, abstractmethod

class AbstractSuper(ABC):
    def delegate(self):
        self.action()

    @abstractmethod
    def action(self):
        pass

class Sub(AbstractSuper):
    def action(self):
        print("action реализован в подклассе")

x = Sub()
x.delegate()

class IncompleteABC(AbstractSuper):
    pass

try:
    IncompleteABC()            # ошибка уже при создании объекта
except TypeError as error:
    print("при создании:", error)

action реализован в подклассе
при создании: Can't instantiate abstract class IncompleteABC without an implementation for abstract method 'action'


### Утиная типизация: общий родитель не обязателен

Функции нужен объект с подходящим поведением. Она может вызвать `read()`
у двух независимых классов, не проверяя их родословную.

Утиная типизация – это то, что `count_words` не проверяет ни тип, ни родителей
объекта: ей подходит любой объект с методом `read()`. Полиморфизм – результат:
один и тот же вызов `source.read()` у разных классов работает по-разному.
Так же было в лекции 5 с `times(x, y)`.

In [3]:
class TextSource:
    def __init__(self, text):
        self.text = text

    def read(self):
        return self.text

class LinesSource:
    def __init__(self, lines):
        self.lines = lines

    def read(self):
        return "\n".join(self.lines)

def count_words(source):
    return len(source.read().split())

print(count_words(TextSource("Доброе утро")))
print(count_words(LinesSource(["Один два", "три"])))

2
3


### Совпадения имени метода недостаточно

Наш `count_words` ожидает `read()` **без аргументов**, возвращающий строку.
Метод с тем же именем, но другим поведением этому интерфейсу не соответствует.
ABC проверяет только, что метод есть, а не что он принимает и возвращает;
поведение проверяют тесты.

In [4]:
class NumberSource:
    def read(self):
        return 42

try:
    count_words(NumberSource())   # 42.split(): у числа нет метода split
except AttributeError:
    print("read() вернул число вместо строки")

assert count_words(TextSource("")) == 0
assert count_words(LinesSource([])) == 0
print("Пустые источники дают ноль слов")

read() вернул число вместо строки
Пустые источники дают ноль слов


### Пространства имён для классов

Имена в Python разрешаются двумя разными механизмами:

- **простые имена** (`X`) – по правилу LEGB, разбиралось в лекции про пространства имён;
- **квалифицированные имена** (`obj.X`) – поиск идёт по пространствам имён объекта
  и его суперклассов, LEGB здесь не участвует вообще.

Тело класса выполняется как обычный блок кода со своей локальной областью: там `X`
находит атрибут класса. Но для методов это тело – **не** enclosing-область: внутри
метода простое имя `X` ищется в самом методе, затем сразу в модуле. Атрибуты класса
метод получает только через `self` или имя класса.


In [53]:
X = "глобальная"

class C:
    X = "атрибут класса"
    print("в теле класса:", X)       # тело класса – своя локальная область

    def method(self):
        print("в методе:", X)         # тело класса пропущено: найдена глобальная X
        print("через self:", self.X)  # атрибут класса – только через объект

C().method()

в теле класса: атрибут класса
в методе: глобальная
через self: атрибут класса


### Классы внутри функций

- Класс можно определить внутри функции, как и `def`. Так пишут функции, которые
  создают и возвращают класс с запомненными параметрами.
- Методу такого класса видна **область функции**: её переменные работают как обычное
  замыкание (лекция 6). А тело самого класса, как на предыдущем экране, методу не видно.


In [63]:
# Функция создаёт класс; методы помнят её параметр greeting
def make_greeter(greeting):
    class Greeter:
        def greet(self, name):
            return f"{greeting}, {name}!"   # greeting – из области функции
    return Greeter

Hello = make_greeter("Привет")         # Hello – класс, созданный функцией
Hi = make_greeter("Здравствуйте")

greeter = Hello()                      # вызов класса создаёт объект
print(greeter.greet("Анна"))           # метод вызывается у объекта, с аргументом
print(Hi().greet("Борис"))             # то же в одну строку

Привет, Анна!
Здравствуйте, Борис!


### Деревья классов и связь пространств имён

По ссылкам `__class__` и `__bases__` (экран «Классы изнутри») можно подняться
от экземпляра по всему дереву наследования. На этом построены инструменты,
которые показывают иерархию классов при отладке.

In [68]:
# Демонстрация того, как можно рекурсивно обойти дерево наследования,
# выводя каждый класс с отступами, отражающими глубину наследования

def classtree(cls, indent):
    """Рекурсивный обход дерева классов"""
    print("." * indent + cls.__name__)
    for supercls in cls.__bases__:
        classtree(supercls, indent + 3)

def instancetree(inst):
    """Показать дерево классов для экземпляра"""
    print("Дерево классов для экземпляра", type(inst).__name__)
    classtree(inst.__class__, 3)

def selftest():
    class A: pass  # MRO: A, object
    class B(A): pass  # B, A, object
    class C(A): pass  # C, A, object
    class D(B, C): pass  # D, B, C, A, object
    class E: pass  # E, object
    class F(D, E): pass  # F, D, B, C, A, E, object

    instancetree(B())
    print()
    instancetree(F())
    # В дереве A дважды (через B и через C), в MRO – один раз
    print("MRO для F:", [cls.__name__ for cls in F.__mro__])


selftest()


Дерево классов для экземпляра B
...B
......A
.........object

Дерево классов для экземпляра F
...F
......D
.........B
............A
...............object
.........C
............A
...............object
......E
.........object
MRO для F: ['F', 'D', 'B', 'C', 'A', 'E', 'object']


### Докстринги (Documentation Strings) в классах и методах

Что такое докстринг и как его прочитать (`__doc__`, `help()`), разбирали в лекции 5
на функциях. У классов механизм тот же, меняется только место:

- в начале **модуля** (файла) – назначение файла;
- в **классе** – роль класса и его ключевые атрибуты и методы;
- в **методе** – что делает, какие параметры принимает и что возвращает.

Первая строка – краткое описание, подробности ниже через пустую строку.
Докстринг описывает назначение, а комментарии `#` – тонкости реализации.


In [69]:
# Докстринги у класса и метода
class ExampleClass:
    """Пример класса с докстрингом.

    Описывает назначение класса и его ключевые методы.
    """

    def method(self, text: str):
        """Печатает текст и возвращает его длину.

        Параметры:
            text (str): исходная строка
        Возвращает:
            int: длина строки
        """
        print(text)
        return len(text)

# Доступ к докстрингам
print("ExampleClass.__doc__:\n", ExampleClass.__doc__)
print("\nExampleClass.method.__doc__:\n", ExampleClass.method.__doc__)

ExampleClass.__doc__:
 Пример класса с докстрингом.

Описывает назначение класса и его ключевые методы.


ExampleClass.method.__doc__:
 Печатает текст и возвращает его длину.

Параметры:
    text (str): исходная строка
Возвращает:
    int: длина строки



## Частые ошибки

- **Изменяемый атрибут класса.** Список или словарь в теле класса – **один на все**
  экземпляры. Изменяемое состояние создают в `__init__` через `self`.
- **Забыли `self` в сигнатуре метода** → `TypeError: takes 0 positional arguments but 1 was given`.
- **Переопределили `__init__` и не вызвали `super().__init__(...)`** → `AttributeError`
  при обращении к полю, которое создаёт родитель (правило – на экране
  «Специализация унаследованных методов»).
- **`type(x) == Class` вместо `isinstance(x, Class)`** – объект подкласса не проходит
  проверку, хотя он тоже `Class` (пример – на следующем экране).


In [ ]:
class Student:
    grades = []                  # ОШИБКА: атрибут КЛАССА – один на всех

    def add(self, grade):
        self.grades.append(grade)

a, b = Student(), Student()
a.add(5)
print("оценки b:", b.grades)     # [5] – хотя b ничего не добавлял


class StudentOk:
    def __init__(self):
        self.grades = []         # атрибут ЭКЗЕМПЛЯРА – у каждого свой

    def add(self, grade):
        self.grades.append(grade)

c, d = StudentOk(), StudentOk()
c.add(5)
print("оценки d:", d.grades)     # [] – как и ожидается


оценки b: [5]
оценки d: []


### `type()` сравнивает только точный класс

В лекции 2 говорили: `isinstance` учитывает наследование, а `type(x) == ...` – нет.
Теперь это можно показать на своих классах. Объект подкласса – это и объект
базового класса, но проверка через `type()` его отвергнет.

In [44]:
class Vehicle:
    pass

class Bus(Vehicle):
    pass

bus = Bus()
print("type(bus) == Vehicle:", type(bus) == Vehicle)          # False: точный класс – Bus
print("isinstance(bus, Vehicle):", isinstance(bus, Vehicle))  # True: автобус тоже транспорт

type(bus) == Vehicle: False
isinstance(bus, Vehicle): True


### Два одноимённых метода не создают перегрузку

В одном теле класса второе определение `__init__` заменяет первое.
Python не выбирает между ними по числу аргументов, поэтому вызов
с одним аргументом падает с `TypeError`.

In [5]:
class Badge:
    def __init__(self, name):
        self.name = name

    def __init__(self, name, role):
        self.name, self.role = name, role

try:
    Badge("Анна")
except TypeError:
    print("Одного аргумента недостаточно: остался второй __init__")

badge = Badge("Анна", "староста")
print(badge.name, badge.role)

Одного аргумента недостаточно: остался второй __init__
Анна староста


### Исправление: один конструктор с необязательным аргументом

Вместо двух `__init__` с предыдущего экрана хватает одного: необязательный аргумент
получает значение по умолчанию. Если способы создания требуют разных форматов
входа, для каждого заводят отдельный фабричный метод, как `from_string` в разделе
методов класса.

In [6]:
class BadgeWithDefault:
    def __init__(self, name, role="участник"):
        self.name, self.role = name, role

for badge in (BadgeWithDefault("Анна"),
              BadgeWithDefault("Борис", "староста")):
    print(badge.name, badge.role)

Анна участник
Борис староста


### Поле экземпляра может перекрыть метод

Здесь флаг и метод получили одинаковое имя `save`.
При чтении `draft.save` найдено поле экземпляра со значением `False`.
Метод остаётся в классе, но обычный вызов через этот экземпляр сломан.

In [4]:
class Draft:
    def __init__(self):
        self.save = False

    def save(self):
        return "Черновик сохранён"

draft = Draft()
print("draft.save:", draft.save)
try:
    draft.save()
except TypeError:
    print("Булево значение нельзя вызвать как метод")
print("Метод в классе остался:", callable(Draft.save))

draft.save: False
Булево значение нельзя вызвать как метод
Метод в классе остался: True


### Разделяем имена состояния и действия

Исправим предыдущий класс: `is_saved` хранит состояние, `save()` выполняет
действие. В этом учебном объекте сохранение только меняет флаг;
записи в файл нет. Проверим состояние до и после вызова.

In [5]:
class DraftFixed:
    def __init__(self):
        self.is_saved = False

    def save(self):
        self.is_saved = True
        return "Черновик сохранён"

draft_fixed = DraftFixed()
print("До вызова:", draft_fixed.is_saved)
print(draft_fixed.save())
print("После вызова:", draft_fixed.is_saved)

До вызова: False
Черновик сохранён
После вызова: True


## Мини-упражнения


### 1. Абстрактный класс и полиморфизм

Опишите абстрактный класс `Shape` с абстрактным методом `area()` и обычным методом
`describe()`, который печатает имя класса и площадь. Сделайте наследников `Rect(w, h)`
и `Circle(r)` и пройдите по ним **одним** циклом.

Проверьте заодно, что создать сам `Shape()` нельзя.


In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

    def describe(self):                     # общий код для всех наследников
        return f"{type(self).__name__}: площадь {self.area():.2f}"


class Rect(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h


class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        return 3.14159 * self.r ** 2


for shape in (Rect(3, 4), Circle(2)):       # полиморфизм: один цикл, разные типы
    print(shape.describe())

try:
    Shape()                                 # абстрактный класс создать нельзя
except TypeError:
    print("Shape() -> TypeError: абстрактный класс создать нельзя")


Rect: площадь 12.00
Circle: площадь 12.57
Shape() -> TypeError: абстрактный класс создать нельзя


### 2. Специализация метода через `super()`

Есть `Employee(name, salary)` с методом `raise_salary(percent)`. Сделайте наследника
`Manager(name, salary, team)`, который:

- в `__init__` вызывает `super().__init__(...)` и добавляет своё поле `team`;
- **расширяет** (а не переписывает) `raise_salary`: менеджеру полагается +5 п.п.
  сверх общей надбавки – вызовите `super().raise_salary(...)`.


In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name, self.salary = name, salary

    def raise_salary(self, percent):
        self.salary = round(self.salary * (1 + percent / 100), 2)
        return self.salary


class Manager(Employee):
    def __init__(self, name, salary, team):
        super().__init__(name, salary)      # инициализируем родительскую часть
        self.team = team

    def raise_salary(self, percent):
        # специализация: менеджеру +5 п.п. сверх общей надбавки
        return super().raise_salary(percent + 5)


staff = [Employee("Иван", 100_000), Manager("Пётр", 100_000, team=["Иван"])]
for person in staff:
    print(f"{person.name:6} {type(person).__name__:9} -> {person.raise_salary(10)}")


Иван   Employee  -> 110000.0
Пётр   Manager   -> 115000.0


### 3. Счётчик экземпляров

Добавьте классу `Employee` подсчёт созданных объектов: атрибут **класса** `count`,
который увеличивается в `__init__`. Выведите `Employee.count` после создания
нескольких сотрудников. Почему здесь атрибут класса уместен – в отличие от
изменяемого списка из «Частых ошибок»?

Подсказка: `type(self).count += 1` и `self.count += 1` ведут себя по-разному.


In [6]:
# Счётчик экземпляров: атрибут класса меняем через type(self)
class Employee:
    count = 0                     # общий счётчик – намеренно общее состояние

    def __init__(self, name):
        self.name = name
        type(self).count += 1     # меняем атрибут класса

for name in ("Иван", "Пётр", "Анна"):
    Employee(name)
print("Employee.count:", Employee.count)


# Ошибка: self.count += 1 читает count из класса,
# а результат записывает в новый атрибут экземпляра
class EmployeeBug:
    count = 0

    def __init__(self, name):
        self.name = name
        self.count += 1

a, b = EmployeeBug("Иван"), EmployeeBug("Пётр")
print("EmployeeBug.count:", EmployeeBug.count)      # класс не изменился
print("a.count:", a.count, "| b.count:", b.count)   # у каждого свой count

Employee.count: 3
EmployeeBug.count: 0
a.count: 1 | b.count: 1


### 4. Полоса прогресса с двумя способами создания

Напишите `Progress(done, total)` для целых чисел: `total > 0`,
`0 <= done <= total`. Некорректные значения должны вызывать `ValueError`.

- `from_string("3/12")` создаёт объект через `cls(...)`.
- Свойство `percent` вычисляет процент выполнения; присваивание ему запрещено.
- `advance(step)` увеличивает выполненный объём только при
  `step > 0` и `done + step <= total`; при ошибке состояние сохраняется.

Проверьте границы, неверную строку, сохранность состояния после отказа
и создание экземпляра подкласса через унаследованный `from_string`.
Типы аргументов считаем заданными: здесь тренируем проверку значений.

### Решение: прогресс и проверка перехода

Сначала проверяем новое значение, затем записываем его.
Расчёт процента использует текущее состояние при каждом чтении.

In [10]:
class Progress:
    def __init__(self, done, total):
        if total <= 0 or not 0 <= done <= total:
            raise ValueError("Недопустимый прогресс")
        self.done, self.total = done, total

    @classmethod
    def from_string(cls, text):
        done, total = text.split("/")
        return cls(int(done), int(total))

    @property
    def percent(self):
        return 100 * self.done / self.total

    def advance(self, step):
        new_done = self.done + step
        if step <= 0 or new_done > self.total:
            raise ValueError("Недопустимый шаг")
        self.done = new_done

### Проверяем границы и сохранность состояния

Проверки относятся к публичным операциям класса. Прямое изменение `done`
или `total` в обход методов не входит в этот контракт.

In [11]:
import pytest

progress = Progress.from_string("3/12")
assert progress.percent == 25
for step in (0, -1, 10):
    with pytest.raises(ValueError):
        progress.advance(step)
    assert progress.done == 3
progress.advance(9)
assert progress.percent == 100
assert Progress(0, 12).percent == 0
for text in ("-1/12", "13/12", "0/0", "0/-1", "abc/12", "3/12/5"):
    with pytest.raises(ValueError):
        Progress.from_string(text)
with pytest.raises(AttributeError):
    progress.percent = 50

class DownloadProgress(Progress):
    pass
assert type(DownloadProgress.from_string("1/4")) is DownloadProgress
print("Границы, отказ без изменений и фабрика подкласса проверены")

Границы, отказ без изменений и фабрика подкласса проверены


## Что почитать

- [Классы](https://docs.python.org/3/tutorial/classes.html) – официальное введение:
  атрибуты, методы, наследование.
- [`abc`](https://docs.python.org/3/library/abc.html) – абстрактные базовые классы.
- М. Лутц, «Изучаем Python» – часть VI, самое подробное изложение ООП на русском.
